In [0]:
from pyspark.sql import functions as F
s3_raw_path = "s3://crypto-lakehouse-raw/raw/"
checkpoint_path = "s3://crypto-lakehouse-raw/_checkpoints/bronze_market/"
schema_path = "s3://crypto-lakehouse-raw/_schemas/bronze_market/"

df_bronze = (spark.readStream\
    .format("cloudFiles")
    .option("cloudFiles.format","json")
    .option("cloudFiles.inferColumnTypes",True)
    .option("cloudFiles.schemaLocation",schema_path)
    .option("multiline",True)
    .load(s3_raw_path)
    .withColumn("ingested_at",F.current_timestamp())
    .withColumn("source_file",F.col("_metadata.file_path"))
)
(
df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation",checkpoint_path)
    .option("mergeSchema","true")
    .trigger(availableNow = True)
    .toTable("crypto_project.crypto.bronze_crypto_market")
    
).awaitTermination()



In [0]:
%sql
SELECT COUNT(*) FROM crypto_project.crypto.bronze_crypto_market 

COUNT(*)
18230
